In [1]:
# Cell 1
# Mục đích: tạo lại client và hàm stats để chạy query có cầu chì (bỏ qua nếu notebook đã có).
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")  # kênh kết nối, giống profiles.yml của dbt

def mb(n):
    # đổi byte sang MB; trả None nếu n là None để lỗi đọc quá sớm lộ ra
    return None if n is None else round(n / 1024 / 1024, 2)

def stats(sql, max_mb=100):
    # tắt cache để số byte phản ánh lần chạy thật; trần byte là cầu chì
    cfg = bigquery.QueryJobConfig(use_query_cache=False, maximum_bytes_billed=max_mb * 1024 * 1024)
    job = client.query(sql, job_config=cfg)
    df = job.result().to_dataframe()  # result() chờ job xong rồi mới đọc byte
    print("processed:", mb(job.total_bytes_processed), "MB | billed:", mb(job.total_bytes_billed), "MB")
    return df

In [2]:
# Cell 2
# Mục đích: lấy 4 con số chuẩn của fct_net_sales để lát nữa đối chiếu với Power BI.
sql = """
SELECT
  COUNT(*)                     AS so_dong,
  COUNT(DISTINCT order_item_id) AS so_id_duy_nhat,
  ROUND(SUM(sale_price), 2)    AS net_revenue,
  ROUND(SUM(gross_profit), 2)  AS gross_profit
FROM `bq-learning-510104.dbt_dev.fct_net_sales`
"""
df = stats(sql)
print(len(df))   # phải là 1; in trước khi dùng df
print(df)

processed: 1.04 MB | billed: 10.0 MB
1
   so_dong  so_id_duy_nhat  net_revenue  gross_profit
0    45529           45529   2685271.28     1391199.0


In [3]:
# Cell 3
# Mục đích: kiểm tra phía "một" của quan hệ: product_id trong dim_products phải duy nhất,
# và mọi product_id của fct_net_sales phải tìm thấy trong dim_products.
sql = """
SELECT
  (SELECT COUNT(*) FROM `bq-learning-510104.dbt_dev.dim_products`) AS so_dong_dim,
  (SELECT COUNT(DISTINCT product_id) FROM `bq-learning-510104.dbt_dev.dim_products`) AS so_product_duy_nhat,
  (SELECT COUNTIF(d.product_id IS NULL)
     FROM `bq-learning-510104.dbt_dev.fct_net_sales` f
     LEFT JOIN `bq-learning-510104.dbt_dev.dim_products` d ON f.product_id = d.product_id) AS fct_khong_khop
"""
df = stats(sql)
print(len(df))
print(df)

processed: 0.57 MB | billed: 20.0 MB
1
   so_dong_dim  so_product_duy_nhat  fct_khong_khop
0        29120                29120               0


In [ ]:
# Cell 4
# Mục đích: xem 10 dòng đầu của dim_products, chỉ 2 cột, không tốn byte quét.
table_id = f"{PROJECT}.dbt_dev.dim_products"
df_preview = client.list_rows(table_id, max_results=10).to_dataframe()
print("Số dòng lấy về:", len(df_preview))
df_preview[["product_id", "department"]]